In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

import optuna, optuna_dashboard
from optuna.trial import TrialState


from src.dataset import *
from src.lightning import *
from src.models import *
from src.params import *
from src.utils import *
from optuna_integration import PyTorchLightningPruningCallback


In [2]:
files_dir = PROCESSED_DIR / "cropped"
metadata = pd.read_csv(DATA_DIR / "train.csv")
dm = BrainDataModule(metadata=metadata, spec_dir= files_dir, batch_size= 32, num_workers= 8, verbose= False)

In [ ]:
def objective(trial, datamodule):

    # def hyperparams to tune

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    weight_decay = 1.2e-5
    dropout =   0.082
    mixup_alpha = 0.33
    n_blocks = 3
    kernel_size = trial.suggest_categorical("kernel_size",[5,7])
    hidden_dims  = [205, 141, 229, 102, 206, 159]


    # def model
    optuna_model = OptunaModel(n_channels= 4, n_classes= 6,
                               hidden_dims= hidden_dims,
                               kernel_size= kernel_size,
                               dropout= dropout)


    # def lightning module
    lit_optuna = BrainLightning(model= optuna_model, n_classes= 6,
                                lr= learning_rate, mixup= True, mixup_alpha= mixup_alpha,
                               scheduler= True, t_max= 20, weight_decay= weight_decay, verbose= False)

    # def callbacks adapted to tuning (short)
    callbacks = [
    ModelCheckpoint(
        dirpath=f"checkpoints/optuna2/{trial.number}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="{epoch:02d}-{val_loss:.3f}",
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        min_delta=1e-3,
    ),
    PyTorchLightningPruningCallback(trial, monitor= "val_loss")
]



    trainer = pl.Trainer(
        max_epochs=20,
        accelerator="auto",
        devices="auto",
        callbacks=callbacks,
        logger= False,
        precision="bf16-mixed",
        enable_progress_bar= False, #silent
        log_every_n_steps=10,
    )

    trainer.fit(lit_optuna, datamodule= datamodule)

    val_loss_final = callbacks[0].best_model_score.item()
    return val_loss_final

In [ ]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_brain_v2.db")
study = optuna.create_study(
    direction = "minimize",
    sampler= optuna.samplers.TPESampler(seed= 273),
    pruner= optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=2 ),
    storage= storage,
    study_name= "brain_optuna_v2",
    load_if_exists= True
)

study.optimize(lambda trial: objective(trial, dm), n_trials= 50, show_progress_bar= True)

trials = study.trials
n_complete = len([t for t in trials if t.state == TrialState.COMPLETE])
n_pruned   = len([t for t in trials if t.state == TrialState.PRUNED])
n_failed   = len([t for t in trials if t.state == TrialState.FAIL])

print(f"Complétés : {n_complete}")
print(f"Pruned    : {n_pruned}")
print(f"Failed    : {n_failed}")

print(f"Best score : {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-04-12 08:42:51,958] Using an existing study with name 'brain_optuna_v2' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[setup] Loading splits from cache: /Users/pablorougerie/code/Projets/brain_waves/data/cache/5_at_seed_273_min_vote_1.json
[setup] Cache loaded — 5 folds found


/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ OptunaModel │  6.8 M │ train │     0 │
│ 1 │ criterion │ KLDivLoss   │      0 │ train │     0 │
└───┴───────────┴─────────────┴────────┴───────┴───────┘

Trainable params: 6.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.8 M                                                                                                
Total estimated model params size (MB): 27                                                                         
Modules in train mode: 48                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 000 | val_loss:   1.4039
Epoch 000 | val_loss:   1.0785


In [ ]:
for ks in [5, 7]:
    m = OptunaModel(n_channels=4, n_classes=6,
                    hidden_dims=[256, 128, 256, 128, 256, 128],
                    kernel_size=ks, dropout=0.082)
    n = sum(p.numel() for p in m.parameters())
    print(f"kernel_size={ks} → {n:,} params")

kernel_size=5 → 4,132,550 params
kernel_size=7 → 8,089,286 params


In [ ]:
import plotly

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_param_importances(study)


In [ ]:
optuna.visualization.plot_parallel_coordinate(study)